# 11 — Trace a run
CrewAI emits execution events. This local listener records event types and
times without capturing prompts or document contents. CrewAI's full tracing
integration can be enabled separately when the organization approves it.



**Basic lesson 11 of 13.**


## Problem

A final answer alone cannot explain tool choice or execution order.


## Concept

CrewAI events expose execution milestones; a local listener records a minimal trace.

Events reveal what ran and in what order. A small local trace is enough to diagnose a missing tool call or unexpected retry without logging the entire customer prompt.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran   ◀ YOU ARE HERE
   12 EVALUATION      checks that fail loudly
```

The flagged row is what this lesson adds. Every other row is the same system, taught in its own lesson.


## Architecture of this lesson

```text
Crew start ──▶ LLM events ──▶ tool events ──▶ Crew end
      │               │                │              │
      └───────────────┴────────────────┴──────────────┘
                         local JSONL trace
```

Crew/LLM/tool events → local JSONL trace → diagnosis. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Register the local CrewAI event listener before kickoff

Register the local CrewAI event listener before kickoff.


In [ ]:
import sys
from pathlib import Path

here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd() / "lesson.py"
sys.path.insert(0, str(next(p for p in here.parents if (p / "pyproject.toml").exists())))

from workshop.common import ensure_runtime
from workshop.crew import build_research_crew
from workshop.observability import LocalTrace

listener = LocalTrace()
result = build_research_crew().kickoff(inputs={
    "question": "Should Northstar AI pilot Meridian AI?",
    "customer_context": "No saved customer constraint.",
})



## Read the local trace
Find the Crew start/end, model calls, and tool events. Compare the event
sequence with the report to diagnose a missing tool or unexpected retry.



## Step 2 — Read the JSONL event sequence and locate the first unexpected step

Read the JSONL event sequence and locate the first unexpected step.


In [ ]:
print(result.raw)
print((ensure_runtime() / "trace.jsonl").read_text(encoding="utf-8")[-3000:])



## Bonus scenario — a cost meter for a tiny game designer
Write your own listener in a few lines (inspired by CrewAI's *Game Builder* crew).
Counting model calls is the simplest cost signal.



In [ ]:
from crewai import Agent, Crew, Process, Task
from crewai.events import BaseEventListener, LLMCallCompletedEvent

from workshop.common import model


class CallCounter(BaseEventListener):
    def __init__(self):
        self.llm_calls = 0
        super().__init__()

    def setup_listeners(self, event_bus):
        @event_bus.on(LLMCallCompletedEvent)
        def count(_source, _event):
            self.llm_calls += 1


counter = CallCounter()
designer = Agent(role="Game Designer", goal="Design tiny games", backstory="You keep rules short.", llm=model(), allow_delegation=False)
rules = Task(
    description="Write the rules of a guess-the-number game in three bullets.",
    expected_output="Three bullets.",
    agent=designer,
)
print(Crew(agents=[designer], tasks=[rules], process=Process.sequential).kickoff().raw)
print("Model calls made:", counter.llm_calls)


## Failure mode to know about

A trace that logs full prompts can leak private documents. This local listener records event types and timestamps; record sensitive payloads only under an explicit data policy.


## Production implication

Record correlation IDs, duration, and boundary outcomes while redacting customer data. Decide whether external trace export is permitted before enabling it.


## Your turn

Find the first model event and each tool event in the trace.

**Checkpoint:** You can reconstruct the key run sequence without reading hidden prompts.
